# Chapter 2

In [8]:
import sys
import os
import torch

# Add project root so we can import the local qwen3.py directly
sys.path.insert(0, os.path.abspath(".."))
import qwen3 as Q

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print(f"Device : {device}")
print(f"PyTorch: {torch.__version__}")

Device : mps
PyTorch: 2.12.1


## 1. Download weights

`download_qwen3_small` fetches two files from HuggingFace (`rasbt/qwen3-from-scratch`) and saves them to the project root:
- `qwen3-0.6B-base.pth` — pretrained weights (~1.2 GB)
- `tokenizer-base.json` — BPE tokenizer config

Already-downloaded files are skipped automatically.

In [2]:
Q.download_qwen3_small(kind="base", out_dir="..")

qwen3-0.6B-base.pth: 100% (1433 MiB / 1433 MiB)


## 2. Tokenizer

`Qwen3Tokenizer` wraps a HuggingFace `tokenizers` BPE model and adds:
- Special-token handling (e.g. `<|im_start|>`, `<|endoftext|>`)
- An optional chat template wrapper (`apply_chat_template=True`)
- EOS token logic that differs between base and reasoning variants

In [3]:
tokenizer = Q.Qwen3Tokenizer("../tokenizer-base.json")

text = "The transformer architecture relies on self-attention."
ids  = tokenizer.encode(text)
back = tokenizer.decode(ids)

print(f"Original : {text!r}")
print(f"Token IDs ({len(ids)} tokens): {ids}")
print(f"Decoded  : {back!r}")
print(f"EOS token: {tokenizer.eos_token!r}  (id={tokenizer.eos_token_id})")

Original : 'The transformer architecture relies on self-attention.'
Token IDs (9 tokens): [785, 42578, 17646, 33644, 389, 656, 12, 53103, 13]
Decoded  : 'The transformer architecture relies on self-attention.'
EOS token: '<|endoftext|>'  (id=151643)


## 3. Model

`QWEN_CONFIG_06_B` holds the architecture hyperparameters. Key differences from a plain GPT-style LLM:
- **GQA** (`n_heads=16`, `n_kv_groups=8`): 8 KV heads shared across 16 query heads — halves KV memory
- **RoPE** (`rope_base=1_000_000`): positional encoding baked into attention, not a separate embedding
- **RMSNorm** instead of LayerNorm: no mean subtraction, slightly cheaper
- **SwiGLU** feed-forward: two parallel linear projections gated with SiLU, then projected down
- **`head_dim=128`** fixed: Q head dim is decoupled from `emb_dim / n_heads`

In [4]:
model = Q.Qwen3Model(Q.QWEN_CONFIG_06_B)

state_dict = torch.load("../qwen3-0.6B-base.pth", map_location="cpu", weights_only=True)
model.load_state_dict(state_dict)
model.to(device)
model.eval()

total  = sum(p.numel() for p in model.parameters())
print(f"Parameters: {total/1e6:.1f}M total")
print(f"dtype     : {next(model.parameters()).dtype}")
print(f"device    : {next(model.parameters()).device}")

Parameters: 751.6M total
dtype     : torch.bfloat16
device    : mps:0


## 4. Generate text

Two-phase autoregressive generation using the KV cache:
1. **Prefill** — process the full prompt in one forward pass; fills the cache for every layer
2. **Decode** — generate one token at a time; each step only computes attention over the new token against cached K/V

In [5]:
@torch.inference_mode()
def generate(model, tokenizer, prompt, max_new_tokens=100, top_k=1, temperature=1.0):
    token_ids    = tokenizer.encode(prompt)
    input_tensor = torch.tensor(token_ids, dtype=torch.long, device=device).unsqueeze(0)

    cache = Q.KVCache(n_layers=model.cfg["n_layers"])
    model.reset_kv_cache()

    # prefill: process the whole prompt in one shot
    logits = model(input_tensor, cache=cache)  # (1, prompt_len, vocab_size)

    generated = list(token_ids)
    for _ in range(max_new_tokens):
        next_logits = logits[:, -1, :]   # prediction for the next position

        if temperature != 1.0:
            next_logits = next_logits / temperature

        if top_k > 1:
            topk_vals, topk_idx = torch.topk(next_logits, top_k)
            probs      = torch.softmax(topk_vals, dim=-1)
            next_token = topk_idx[0, torch.multinomial(probs[0], 1)].item()
        else:
            next_token = next_logits.argmax(dim=-1).item()  # greedy

        if next_token == tokenizer.eos_token_id:
            break

        generated.append(next_token)

        # decode: one token at a time — KV cache skips recomputing past keys/values
        next_input = torch.tensor([[next_token]], dtype=torch.long, device=device)
        logits     = model(next_input, cache=cache)  # (1, 1, vocab_size)

    return tokenizer.decode(generated)

In [7]:
prompt = "The key idea behind transformers is"
print(f"Prompt:\n{prompt}\n")
print("Output:")
print(generate(model, tokenizer, prompt, max_new_tokens=100, top_k=1))

Prompt:
The key idea behind transformers is

Output:
The key idea behind transformers is that the input and output of the transformer are the same. This is because the transformer is a linear operator, and linear operators preserve the structure of the input space. In other words, the output of the transformer is a linear combination of the input, and the input is a linear combination of the output. This is why the input and output of the transformer are the same.
The key idea behind transformers is that the input and output of the transformer are the same. This is because the transformer is a
